In [1]:
from __future__ import annotations

import os
import re
import json
import math
import random
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import weight_norm
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor, get_cosine_schedule_with_warmup
from accelerate import Accelerator

In [2]:
import sys
sys.path.append('/workspace/MedVQA-Classification/src')

In [3]:
from data.dataloaders import *

In [4]:
from __future__ import annotations

from dataclasses import dataclass, fields
from pathlib import Path
from typing import Optional, Any, Dict

import yaml


@dataclass
class TrainConfig:
    dataset_name: str = "flaviagiammarino/vqa-rad"
    model_name: str = "flaviagiammarino/pubmed-clip-vit-base-patch32"
    model_type: str = "pubmedclip_ban"

    output_dir: str = "runs/pubmedclip_ban"
    seed: int = 42

    epochs: int = 40
    batch_size: int = 16
    eval_batch_size: int = 64
    num_workers: int = 2
    max_length: int = 77

    num_hid: int = 1536
    glimpse: int = 4
    freeze_clip: bool = True

    lr_head: float = 1e-4
    lr_clip: float = 1e-6
    weight_decay: float = 1e-4
    warmup_ratio: float = 0.05
    type_loss_weight: float = 0.2
    class_weight_power: float = 0.5
    mixed_precision: str = "no"
    grad_clip: float = 1.0

    min_answer_freq: int = 1
    max_answers: Optional[int] = None
    answer_vocab_source: str = "train_eval"
    filter_eval_unseen_train_answers: bool = False

    filter_invalid_count_answers: bool = False
    count_answer_max_num: Optional[int] = None

    eval_with_type_mask: bool = True


def load_config(path: str | Path, overrides: Optional[Dict[str, Any]] = None) -> TrainConfig:
    path = Path(path)
    data = yaml.safe_load(path.read_text(encoding="utf-8")) or {}

    if overrides:
        for key, value in overrides.items():
            if value is not None:
                data[key] = value

    valid_fields = {f.name for f in fields(TrainConfig)}
    unknown = set(data) - valid_fields
    if unknown:
        raise ValueError(f"Unknown config keys: {sorted(unknown)}")

    return TrainConfig(**data)


In [5]:
class FCNet(nn.Module):
    def __init__(self, dims: List[int], act: str = "ReLU", dropout: float = 0.2):
        super().__init__()
        layers = []
        for i in range(len(dims) - 1):
            layers.append(weight_norm(nn.Linear(dims[i], dims[i + 1],bias=False), name="weight", dim=None))
            if act:
                layers.append(getattr(nn, act)())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        self.main = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.main(x)


class BCNet(nn.Module):
    def __init__(self, v_dim: int, q_dim: int, h_dim: int, h_out: Optional[int] = None,
                 dropout: Tuple[float, float] = (0.2, 0.5), k: int = 3):
        super().__init__()
        self.h_dim = h_dim
        self.h_out = h_out
        self.k = k
        self.v_net = FCNet([v_dim, h_dim * k], act="ReLU", dropout=dropout[0])
        self.q_net = FCNet([q_dim, h_dim * k], act="ReLU", dropout=dropout[0])
        self.dropout = nn.Dropout(dropout[1])
        self.p_net = nn.AvgPool1d(k, stride=k) if k > 1 else None

        if h_out is not None:
            self.h_mat = nn.Parameter(torch.randn(1, h_out, 1, h_dim * k) * 0.01)
            self.h_bias = nn.Parameter(torch.zeros(1, h_out, 1, 1))

    def forward(self, v: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
        v_ = self.dropout(self.v_net(v))
        q_ = self.q_net(q)
        if self.h_out is None:
            return torch.einsum("bvk,bqk->bvqk", v_, q_)
        return torch.einsum("ghyk,bvk,bqk->bhvq", self.h_mat, v_, q_) + self.h_bias

    def forward_with_weights(self, v: torch.Tensor, q: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
        v_ = self.v_net(v)
        q_ = self.q_net(q)
        logits = torch.einsum("bvk,bvq,bqk->bk", v_, w, q_)
        if self.k > 1:
            logits = self.p_net(logits.unsqueeze(1)).squeeze(1) * self.k
        return logits


class BiAttention(nn.Module):
    def __init__(self, v_dim: int, q_dim: int, h_dim: int, glimpse: int = 4):
        super().__init__()
        self.glimpse = glimpse
        self.logits = BCNet(v_dim, q_dim, h_dim, h_out=glimpse, dropout=(0.2, 0.5), k=3)

    def forward_all(self, v: torch.Tensor, q: torch.Tensor, q_mask: Optional[torch.Tensor] = None):
        B, V, Q = v.size(0), v.size(1), q.size(1)
        logits = self.logits(v, q)
        if q_mask is not None:
            mask = q_mask[:, None, None, :].bool()
            logits = logits.masked_fill(~mask, -1e4)
        att = F.softmax(logits.reshape(B, self.glimpse, V * Q), dim=-1).reshape(B, self.glimpse, V, Q)
        return att, logits


class SimpleClassifier(nn.Module):
    def __init__(self, in_dim: int, hid_dim: int, out_dim: int, dropout: float = 0.5):
        super().__init__()
        self.main = nn.Sequential(
            weight_norm(nn.Linear(in_dim, hid_dim,bias=False), name="weight", dim=None),
            nn.ReLU(),
            nn.Dropout(dropout),
            weight_norm(nn.Linear(hid_dim, out_dim,bias=False), name="weight", dim=None),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.main(x)


class PubMedCLIPBANFixed(nn.Module):
    def __init__(
        self,
        clip_model: CLIPModel,
        num_answers: int,
        num_hid: int = 512,
        glimpse: int = 4,
        freeze_clip: bool = True,
    ):
        super().__init__()
        self.clip = clip_model
        self.freeze_clip = freeze_clip
        self.v_dim = clip_model.config.vision_config.hidden_size
        self.q_dim = clip_model.config.text_config.hidden_size
        self.glimpse = glimpse

        if freeze_clip:
            for p in self.clip.parameters():
                p.requires_grad = False

        self.v_att = BiAttention(self.v_dim, self.q_dim, num_hid, glimpse=glimpse)
        self.b_net = nn.ModuleList([
            BCNet(self.v_dim, self.q_dim, num_hid, h_out=None, dropout=(0.2, 0.5), k=3)
            for _ in range(glimpse)
        ])
        self.q_prj = nn.ModuleList([
            FCNet([num_hid, self.q_dim], act="ReLU", dropout=0.2)
            for _ in range(glimpse)
        ])

        self.answer_classifier = SimpleClassifier(self.q_dim, num_hid * 2, num_answers, dropout=0.5)
        self.type_classifier = SimpleClassifier(self.q_dim, num_hid, 2, dropout=0.3)

    def train(self, mode: bool = True):
        super().train(mode)
        if self.freeze_clip:
            self.clip.eval()
        return self

    def encode_tokens(self, pixel_values, input_ids, attention_mask):
        grad_enabled = any(p.requires_grad for p in self.clip.parameters())
        with torch.set_grad_enabled(grad_enabled):
            vision_outputs = self.clip.vision_model(pixel_values=pixel_values, return_dict=True)
            text_outputs = self.clip.text_model(
                input_ids=input_ids, attention_mask=attention_mask, return_dict=True
            )
        # ViT: bỏ CLS token; ResNet CLIP có thể cần chỉnh lại nếu dùng RN backend.
        visual_tokens = vision_outputs.last_hidden_state[:, 1:, :]
        text_tokens = text_outputs.last_hidden_state
        return visual_tokens, text_tokens

    def forward(self, pixel_values, input_ids, attention_mask):
        v, q = self.encode_tokens(pixel_values, input_ids, attention_mask)
        att, _ = self.v_att.forward_all(v, q, q_mask=attention_mask)

        for g in range(self.glimpse):
            b_emb = self.b_net[g].forward_with_weights(v, q, att[:, g])
            q = q + self.q_prj[g](b_emb).unsqueeze(1)

        mask = attention_mask.unsqueeze(-1).float()
        q_pooled = (q * mask).sum(1) / mask.sum(1).clamp_min(1.0)

        answer_logits = self.answer_classifier(q_pooled)
        type_logits = self.type_classifier(q_pooled)
        return answer_logits, type_logits, att


def mask_logits_by_predicted_type(
    logits: torch.Tensor,
    type_logits: torch.Tensor,
    label_type_ids: torch.Tensor,
    mask_value: float = -1e4,
) -> torch.Tensor:
    """
    label_type_ids: [num_answers], 0=open, 1=closed
    type_logits -> predicted type per sample.
    """
    pred_type = type_logits.argmax(dim=-1)  # [B]
    allowed = label_type_ids.unsqueeze(0).to(logits.device) == pred_type.unsqueeze(1)
    return logits.masked_fill(~allowed, mask_value)

In [6]:
from engine.evaluate import evaluate

In [7]:
from engine.trainer import train_main

In [8]:
# from collections import Counter
# # def train_main(cfg: TrainConfig):
# #     # seed_everything(cfg.seed)

# #     accelerator = Accelerator(mixed_precision=cfg.mixed_precision)
# #     if accelerator.is_main_process:
# #         os.makedirs(cfg.output_dir, exist_ok=True)
# #         with open(os.path.join(cfg.output_dir, "config.json"), "w", encoding="utf-8") as f:
# #             json.dump(asdict(cfg), f, indent=2, ensure_ascii=False)
# #     falsed_sample=Counter()
# #     processor = CLIPProcessor.from_pretrained(cfg.model_name)
# #     data_bundle = make_loaders(cfg, processor)
# #     train_loader = data_bundle["train_loader"]
# #     eval_loader = data_bundle["eval_loader"]
# #     test_loader = data_bundle["test_loader"]
# #     label2ans = data_bundle["label2ans"]
# #     ans2label = data_bundle["ans2label"]
# #     label_type_ids = data_bundle["label_type_ids"]
# #     class_weights = data_bundle["class_weights"]
# #     train_answer_set = data_bundle["train_answer_set"]
# #     filter_stats = data_bundle["filter_stats"]
# #     if accelerator.is_main_process:
# #         print(
# #             "[count-filter] "
# #             f"enabled={filter_stats['filter_invalid_count_answers']} | "
# #             f"max_num={filter_stats['count_answer_max_num']} | "
# #             f"train {filter_stats['train_original_len']} -> {filter_stats['train_filtered_len']} "
# #             f"(invalid_count={filter_stats['train_invalid_count_answer_samples']}) | "
# #             f" {filter_stats['eval_original_len']} -> {filter_stats['eval_filtered_len']} "
# #             f"(invalid_count={filter_stats['eval_invalid_count_answer_samples']})"
# #         )

# #     clip_model = CLIPModel.from_pretrained(cfg.model_name)
# #     model = PubMedCLIPBANFixed(
# #         clip_model=clip_model,
# #         num_answers=len(label2ans),
# #         num_hid=cfg.num_hid,
# #         glimpse=cfg.glimpse,
# #         freeze_clip=cfg.freeze_clip,
# #     )
# #     head_params, clip_params = [], []
# #     for n, p in model.named_parameters():
# #         if not p.requires_grad:
# #             continue
# #         if n.startswith("clip."):
# #             clip_params.append(p)
# #         else:
# #             head_params.append(p)

# #     optimizer = torch.optim.AdamW(
# #         [
# #             {"params": head_params, "lr": cfg.lr_head},
# #             {"params": clip_params, "lr": cfg.lr_clip},
# #         ],
# #         weight_decay=cfg.weight_decay,
# #     )
# #     total_steps = cfg.epochs * math.ceil(len(train_loader) / max(accelerator.num_processes, 1))
# #     warmup_steps = int(total_steps * cfg.warmup_ratio)
# #     scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# #     model, optimizer, train_loader, test_loader, scheduler = accelerator.prepare(
# #         model, optimizer, train_loader, test_loader, scheduler
# #     )
# #     class_weights = class_weights.to(accelerator.device)
# #     label_type_ids = label_type_ids.to(accelerator.device)

# #     best = -1.0
# #     best_path = os.path.join(cfg.output_dir, "best_model.pt")
# #     for epoch in range(1, cfg.epochs + 1):
# #         model.train()
# #         loss_meter = 0.0
# #         step_count = 0

# #         for batch in train_loader:
# #             optimizer.zero_grad(set_to_none=True)
# #             answer_logits, type_logits, _ = model(
# #                 pixel_values=batch["pixel_values"],
# #                 input_ids=batch["input_ids"],
# #                 attention_mask=batch["attention_mask"],
# #             )

# #             answer_loss = F.cross_entropy(
# #                 answer_logits,
# #                 batch["labels"],
# #                 weight=class_weights,
# #                 ignore_index=-100,
# #             )
# #             type_loss = F.cross_entropy(type_logits, batch["answer_type"])
# #             loss = answer_loss + cfg.type_loss_weight * type_loss

# #             accelerator.backward(loss)
# #             if cfg.grad_clip and cfg.grad_clip > 0:
# #                 accelerator.clip_grad_norm_(model.parameters(), cfg.grad_clip)
# #             optimizer.step()
# #             scheduler.step()

# #             loss_meter += accelerator.gather_for_metrics(loss.detach()).mean().item()
# #             step_count += 1
# #         metrics = evaluate(model, test_loader, accelerator, label_type_ids, cfg)
# #         falsed_sample.update(metrics['falsed_samples'])
# #         if accelerator.is_main_process:
# #             score = metrics["overall_acc"]
# #             if score > best:
# #                 best = score
# #                 unwrapped = accelerator.unwrap_model(model)
# #                 torch.save(
# #                     {
# #                         "model": unwrapped.state_dict(),
# #                         "label2ans": label2ans,
# #                         "ans2label": ans2label,
# #                         "label_type_ids": label_type_ids.detach().cpu(),
# #                         "train_answer_set": sorted(list(train_answer_set)),
# #                         "answer_vocab_source": cfg.answer_vocab_source,
# #                         "filter_stats": filter_stats,
# #                         "config": asdict(cfg),
# #                     },
# #                     best_path,
# #                 )

# #             print(
# #                 f"Epoch {epoch:03d}/{cfg.epochs} | "
# #                 f"loss={loss_meter / max(step_count, 1):.4f} | "
# #                 f"overall={metrics['overall_acc']:.4f} | "
# #                 f"open={metrics['open_acc']:.4f} | "
# #                 f"closed={metrics['closed_acc']:.4f} | "
               
# #                 f"best={best:.4f}"
# #             )

# #     accelerator.wait_for_everyone()
# #     print(falsed_sample)
# #     return best_path,falsed_sample


In [9]:
from accelerate import notebook_launcher

# cfg = TrainConfig(
#     dataset_name="flaviagiammarino/vqa-rad",
#     model_name="flaviagiammarino/pubmed-clip-vit-base-patch32",
#     output_dir="./medvqa_runs/vqa_rad_pubmedclip_ban",
#     epochs=200,
#     batch_size=32,
#     lr_head=1e-4,
#     lr_clip=1e-5,
#     freeze_clip=True,
#     answer_vocab_source="train_eval",  # repo/paper-compatible. Dùng "train" cho strict protocol.
#     mixed_precision="bf16",
# )
cfg = TrainConfig(dataset_name= '/workspace/MedVQA-Classification/datasets/slake',epochs=1)
notebook_launcher(train_main, args=(cfg,), num_processes=1)

Launching training on one GPU.


[count-filter] enabled=False | max_num=None | train 4919 -> 4919 (invalid_count=0) | 1053 -> 1053 (invalid_count=0)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Epoch 001/1 | loss=2.4587 | test_overall=0.5156 | test_open=0.5070 | test_closed=0.5288 | val_overall=0.5404 | val_open=0.5388 | val_closed=0.5427 | test_best=0.5156


In [12]:
from data.data_prep import PreDataset



len(PreDataset('/workspace/MedVQA-Classification/datasets/slake','train'))

4919